# 02 · Atölye modeli: işler, durumlar, termin ve duruş

Benchmark'tan sonra gerçek atölye senaryosu. Bütün girdiler `core.domain_rules` içindeki Pydantic
modellerinden geçer; kurala uymayan veri çözücüye ulaşmadan reddedilir.

Bu notebook'ta:
1. Demo atölyeyi (`data/examples/order_demo.json`) okumak
2. Tamamlanmış / devam eden / başlamamış operasyonları tanımak
3. Hatalı girdinin nasıl reddedildiğini görmek
4. Senaryoyu çözmek ve doğrulamak
5. Makine duruşu eklemek; retrospektif rotanın neden engellendiğini görmek

> **Bütün süreler ve terminler sentetik demo değeridir.** Zaman birimi t=0 referansından tamsayı dakikadır.

In [1]:
# Notebook'u tutorials/ veya repo kökünden açabilirsiniz; importlar repo kökünden çalışır.
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "tutorials":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import plotly.io as pio
pio.renderers.default = "notebook_connected"   # plotly.js'i her notebook'a gömmez; görüntülemek için internet gerekir
print("Repo kökü:", ROOT)

Repo kökü: /Users/nazlsu/Desktop/PickWise


In [2]:
import pandas as pd

from benchmarks.workshop_order_demo import DemoInput
from core.cpsat_solver import solve
from core.domain_rules import OperationModel, WorkshopScenario
from core.workshop_fjsp_builder import build_env

demo = DemoInput.model_validate_json(Path("data/examples/order_demo.json").read_text(encoding="utf-8"))
scenario = demo.baseline
print(scenario.description)
print("Şu an (current_time):", scenario.current_time, "dk")

Sentetik atölye örneği; bütün süreler ve terminler demo varsayımıdır. Sürekli çalışma kabul edilir.
Şu an (current_time): 5 dk


## 1. Makineler ve işler

In [3]:
machines = {m.machine_id: m.name for m in scenario.machines}
rows = []
for job in scenario.jobs:
    for op in job.operations:
        rows.append({
            "İş": job.name,
            "Operasyon": op.name,
            "Durum": op.status.value,
            "Uygun makineler (dk)": ", ".join(f"{machines[m]}: {d}" for m, d in op.processing_times.items()),
            "Sabit başlangıç": op.fixed_start,
            "Sabit bitiş": op.fixed_end,
            "Kalan süre": op.remaining_duration,
            "Hazır olma": job.release_time,
            "Yumuşak termin": job.due_date,
            "Sert termin": job.deadline,
        })
pd.DataFrame(rows).convert_dtypes()

,İş,Operasyon,Durum,Uygun makineler (dk),Sabit başlangıç,Sabit bitiş,Kalan süre,Hazır olma,Yumuşak termin,Sert termin
0,Devam eden iş,Tamamlanmış işlem,completed,Torna 1: 3,0,3,<NA>,0,10,None
1,Devam eden iş,Devam eden işlem,in_progress,Torna 1: 7,3,<NA>,5,0,10,None
2,Bekleyen mevcut iş,Tornalama,not_started,Torna 1: 5,<NA>,<NA>,<NA>,10,20,None


- **completed:** makinesi, başlangıcı ve bitişi sabittir; yeniden planlanmaz.
- **in_progress:** makinesi ve başlangıcı sabit, `remaining_duration` kadar daha sürer.
- **not_started:** çözücü makine ve başlangıç seçer; `current_time` öncesine yerleştirilmez.
- `due_date` **yumuşak termindir** (raporlanır), `deadline` **sert termindir** (çözücüye kısıt).

## 2. Hatalı girdi reddedilir

In [4]:
from pydantic import ValidationError

samples = {
    "Başlamamış işleme sabit başlangıç": dict(operation_id=1, name="x", processing_times={100: 5}, fixed_start=2),
    "Devam eden işlemde kalan süre yok": dict(operation_id=1, name="x", processing_times={100: 5},
                                              status="in_progress", fixed_machine_id=100, fixed_start=2),
    "Sıfır süre": dict(operation_id=1, name="x", processing_times={100: 0}),
}
for label, data in samples.items():
    try:
        OperationModel.model_validate(data)
    except ValidationError as error:
        print(f"{label}: {error.errors()[0]['msg']}")

Başlamamış işleme sabit başlangıç: Value error, NOT_STARTED cannot have fixed/remaining fields
Devam eden işlemde kalan süre yok: Value error, IN_PROGRESS requires remaining_duration and no fixed_end
Sıfır süre: Input should be greater than 0


## 3. Senaryoyu çözmek

In [5]:
import plotly.graph_objects as go

def gantt(rows, title, machine_label, job_label, x_title):
    """rows: çizelge satırları (job, operation, machine, start, end)."""
    fig = go.Figure()
    for j in sorted({r["job"] for r in rows}):
        group = [r for r in rows if r["job"] == j]
        fig.add_trace(go.Bar(
            name=job_label(j), orientation="h",
            y=[machine_label(r["machine"]) for r in group],
            x=[r["end"] - r["start"] for r in group],
            base=[r["start"] for r in group],
            text=[f"O{r['operation'] + 1}" for r in group],
            hovertemplate="%{y}<br>Başlangıç: %{base}<br>Süre: %{x}<extra>%{fullData.name}</extra>",
        ))
    fig.update_layout(title=title, barmode="overlay", xaxis_title=x_title,
                      yaxis_title="Makine", height=420, legend_title="İş")
    fig.update_yaxes(categoryorder="category descending")
    return fig

In [6]:
def solve_scenario(s, time_limit=10):
    env, _, _ = build_env(s)
    return solve(env, time_limit=time_limit)

result = solve_scenario(scenario)
print("Durum:", result["status"], "| makespan:", result["makespan"], "dk | ihlal:", result["violations"])

names = {j: job.name for j, job in enumerate(scenario.jobs)}
order = [m.machine_id for m in scenario.machines]
gantt(result["schedule"], "Başlangıç planı",
      machine_label=lambda m: machines[order[m]], job_label=lambda j: names[j],
      x_title="t=0 referansından dakika")

Durum: OPTIMAL | makespan: 15 dk | ihlal: []


`solve` atölye senaryosunda çizelgeyi `validate_workshop` ile kendisi denetler; ihlal varsa hata verir.
Sonuç satırlarındaki `source_*_id` alanları orijinal kimlikleri taşır (`job`, `machine` iç indistir).

## 4. Makine duruşu

Torna 1 için t=10–14 arasında plansız duruş tanımlayalım. Operasyonlar bölünemez; bekleyen iş duruştan sonraya kayar.

In [7]:
data = scenario.model_dump()
data["scenario_id"] = "workshop_demo_downtime"
data["machines"][0]["unavailable_intervals"] = [(10, 14)]
with_downtime = WorkshopScenario.model_validate(data)

after = solve_scenario(with_downtime)
print("Durum:", after["status"], "| makespan:", result["makespan"], "→", after["makespan"], "dk")
gantt(after["schedule"], "Torna 1 duruşu (t=10–14) ile",
      machine_label=lambda m: machines[order[m]], job_label=lambda j: names[j],
      x_title="t=0 referansından dakika")

Durum: OPTIMAL | makespan: 15 → 19 dk


Duruş kısıtı tek başına "olay yönetimi" değildir; duruş için ayrı önce/sonra servisi sonraki iş paketidir (WP-4A).

## 5. Retrospektif rota engellenir

Geçmiş kayıtlardan çıkarılmış rota, kullanıcı onayı olmadan kesin planlamaya giremez.

In [8]:
data = scenario.model_dump()
data["jobs"][1]["route_source"] = "retrospective"
retro = WorkshopScenario.model_validate(data)
try:
    build_env(retro)
except ValueError as error:
    print("Engellendi:", error)

env, _, _ = build_env(retro, approved_only=False)   # açıkça izin verilirse çalışır ve etiketlenir
print("Etiket:", solve(env, 10)["scenario_type"])

Engellendi: RETROSPECTIVE routes require user approval; use approved_only=False only for retrospective simulation
Etiket: retrospective
